# Capability: validation

The `validation` capability checks that extracted values satisfy constraints defined in the contract (e.g., min/max length, regex pattern conformance, enum membership).

Validation runs **after** extraction — the reconciler uses it to grade candidates.

In [ ]:
from decimal import Decimal
from pydantic import BaseModel, Field

import paxman
import paxman.contract.adapters.pydantic  # noqa: F401


class ValidatedOrder(BaseModel):
    order_id: str = Field(..., min_length=5, max_length=20)
    amount: Decimal = Field(..., gt=Decimal("0"), lt=Decimal("10000"))
    status: str = Field(..., pattern=r"^(pending|approved|shipped)$")


sample = """
Order: ORD-001
Amount: 1500.00
Status: approved
"""

result = paxman.normalize(sample, ValidatedOrder)
print(f"Status: {result.status.name}")
print(f"Data:   {result.normalized_data}")
print(f"Unresolved: {result.unresolved_fields}")

In [ ]:
# Validation failure example — status doesn't match the allowed pattern
bad_sample = """
Order: ORD-002
Amount: 200.00
Status: CANCELLED
"""

bad_result = paxman.normalize(bad_sample, ValidatedOrder)
print(f"Status: {bad_result.status.name}")
print(f"Data:   {bad_result.normalized_data}")
if bad_result.diagnostics:
    for d in bad_result.diagnostics:
        print(f"  Diag: {d.code.name} — {d.message}")

In [ ]:
# Boundary test: amount just outside range
boundary_sample = """
Order: ORD-003
Amount: 10001.00
Status: pending
"""

boundary_result = paxman.normalize(boundary_sample, ValidatedOrder)
print(f"Status: {boundary_result.status.name}")
print(f"Unresolved: {boundary_result.unresolved_fields}")
for d in boundary_result.diagnostics:
    print(f"  Diag: {d.code.name} — {d.message}")

## How it works

1. The **planner** reads `CanonicalField` constraints from the adapted contract.
2. The **executor** runs `validation` as a post-extraction step for each field.
3. The **reconciler** uses validation results to adjust confidence — a validated value is more trustworthy.
4. Values that fail validation still appear in the artifact, but with lower confidence and a diagnostic.

The `validation` capability is **always the last capability** in the execution chain for each field.

## Try it yourself

- Add an `email` field with `pattern` validation and test against valid/invalid emails.
- What happens when a field passes `regex_extraction` but fails `validation`? (The reconciler produces an `UNTRUSTED` candidate.)
- Test `Decimal` boundary conditions: `0`, negative values, very large values.